# AirWindow 2.1: reproducible historical decision replay

Run in Colab or local Jupyter after cloning `https://github.com/acchasujal/aqi` and opening this notebook. This notebook uses committed real retrospective OOF replay artifacts. It does not fit a model or fetch live data. Historical issue times are simulated fixed origins; first-publication times remain unknown. No calibrated probability or confidence interval is claimed.

In Google Colab, first run the setup cell below. It clones public source only if no checkout exists. No login, paid compute, raw targets or live data API is required. Local verification is distinct from a Google-hosted run.


In [ ]:
from pathlib import Path
import subprocess
if not any((p/"apps/airwindow/public/data/replay.json").exists() for p in [Path.cwd(), *Path.cwd().parents, Path.cwd()/"aqi"]):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/acchasujal/aqi.git", "aqi"], check=True)


In [ ]:
from pathlib import Path
import json, hashlib, sys
import pandas as pd
root = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd()/"aqi"] if (p/"apps/airwindow/public/data/replay.json").exists()), None)
if root is None:
    raise FileNotFoundError("Clone acchasujal/aqi and run this notebook from its checkout")
sys.path.insert(0, str(root))
data = root/"apps/airwindow/public/data"
replay = json.loads((data/"replay.json").read_text())
manifest = json.loads((data/"manifest.json").read_text())
assert hashlib.sha256((data/"replay.json").read_bytes()).hexdigest() == manifest["replay_sha256"]


## Recompute every eligible case

The predeclared school scenario is original 09:00, feasible 08:00?18:00, durations 1/2/3h, both target days and all four stations. Every forecast window is ranked before observed outcome availability is inspected. Missing paired outcomes remain in coverage. Cases from one origin and overlapping durations are dependent; counts are not independent trials. Hindsight is calculated only where all feasible outcomes are observed.


In [ ]:
from apps.airwindow.replay_validation import validate_replay
from apps.airwindow.build_artifacts import metric, origin_bootstrap
validate_replay(replay)
cases = pd.DataFrame(replay["cases"])
summary = metric(cases)
for key, value in summary.items():
    reference = replay["summary"][key]
    if isinstance(value, float):
        assert abs(value-reference) < 1e-10
    else:
        assert value == reference
summary


In [ ]:
tables=[]
for (station, duration, split), part in cases.groupby(["station_id", "duration_hours", "split"]):
    tables.append({"station_id":station,"duration_hours":duration,"split":split, **metric(part)})
pd.DataFrame(tables)


## Chronological confirmation and contiguous-origin resampling

Development errors inform descriptive comparison states on later origins only. Later origins were previously inspected by analysts, so this is not independent calibration. Exploratory resampling uses whole paired cases in contiguous four-origin blocks. No resampling interval should be interpreted as live decision confidence. No robust recommendations are made; success fractions describe numerical retrospective choices.


In [ ]:
for split in ["development","confirmation"]:
    part=cases[cases["split"].eq(split)]
    print(split, metric(part))
    print(origin_bootstrap(part))
assert replay["calibration"]["status"] == "not_calibrated"
assert replay["summary"]["recommended_cases"] == 0
assert not cases.decision_status.eq("ROBUST IMPROVEMENT").any()


## Inspect an origin without choosing a success

The default is the earliest retained case, selected by fixed case-ID ordering. Change the case ID to inspect another historical scenario. Forecast and observed columns are distinct. Negative selected-minus-original differences mean lower concentration; they are not measured health or exposure benefits.


In [ ]:
case = sorted(replay["cases"], key=lambda x:x["case_id"])[0]
print({k:case[k] for k in ["case_id","station_name","issue_time_ist","pm_history_cutoff_ist","original_start_ist","selected_start_ist","predicted_difference","observed_difference","decision_status"]})
pd.DataFrame(case["hours"])
